In [ ]:
import pickle

import sys
sys.path.append('..')

import jax
import jax.numpy as jnp
import numpy as np
from matplotlib import pyplot as plt

from lucid.geometry import generate_detector
from lucid.generate import read_photon_data_from_photonsim
from lucid.optimization.grid_search import load_optimization_config
from lucid.detector_params import ParticleParams, load_detector_params

# New likelihood imports
from lucid.losses import (
    first_arrival_nll,
    poisson_nll,
    origin_time_loss_configurable,
    TAU_VTX_PARAM_A,
    TAU_VTX_PARAM_B,
    TAU_VTX_PARAM_C,
)

PHYSICS_CONFIG = '../config/JUNO_physics_config.json'

print("Imports successful")

In [ ]:
from lucid.simulation import setup_event_simulator

# =====================================================================
# Full script: 2D parameter scans + optimization trajectory overlay
# =====================================================================
import time
import pickle
from pathlib import Path
import numpy as np
import jax
import jax.numpy as jnp
from jax import value_and_grad
from tqdm import tqdm
import matplotlib.pyplot as plt
import json
import os

# =====================================================================
# Configuration
# =====================================================================
det_json_filename = '../config/JUNO_geom_config.json'
# Load the JSON file
with open(det_json_filename, 'r') as f:
    det_config = json.load(f)
detector_type = det_config["detector_type"].capitalize()
basename = os.path.basename(det_json_filename)
detector_name = basename.split('_')[0]

TEMPERATURE = 0.10
K = 7
Nphot = 150_000

C_MEDIUM = 0.299792 / 1.33  # speed of light in medium

# =====================================================================
# Detector setup
# =====================================================================
detector = generate_detector(det_json_filename)
detector_points = jnp.array(detector.all_points)
detector_radius = detector.S_radius
NUM_DETECTORS = len(detector_points)

prediction_simulator = setup_event_simulator(
    det_json_filename, Nphot, TEMPERATURE, 
    max_sensors_per_cell=4, K=K, is_data=False, hit_mode='per_photon',
    detector_type=detector_type,
    physics_config=PHYSICS_CONFIG, default_detector_params=True
)

# Setup data simulator for generating target events (is_data=True, temperature=0.0)
data_simulator = setup_event_simulator(det_json_filename, Nphot, temperature=0.0, K=20,
                                      is_data=True, is_calibration=False, detector_type=detector_type,
                                      physics_config=PHYSICS_CONFIG, default_detector_params=True)

In [ ]:
import jax
import jax.numpy as jnp
import time

from jax import jit
from pathlib import Path

from matplotlib import pyplot as plt
plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 12

import numpy as np
from functools import partial
import pickle
from tqdm import tqdm
from jax import grad, jit, vmap, value_and_grad
import uproot
from scipy.interpolate import interp1d
import subprocess

from lucid.optimization.run import load_config
from lucid.optimization.grid_search import get_detector_bounds

# =====================================================================
# Likelihood-based loss with vertex term and dynamic tau_vtx
# =====================================================================
TAU_TIME = 0.15  # Fixed tau for first_arrival_nll
NRAYS_FLOAT = float(Nphot)  # From Cell 2


@jit
def combined_product_loss(params, observed_times, observed_counts, key):
    """
    Likelihood-based loss: 3-term geometric mean of charge, time, and vertex losses.

    Args:
        params: [x, y, z, t0, theta, phi, energy]

    Returns:
        combined_loss, (charge_loss, time_loss, vertex_loss)
    """
    position = params[:3]
    t0 = params[3]
    theta = params[4]
    phi = params[5]
    energy = params[6]

    # Simulate with t0=0
    track = ParticleParams(energy=energy, position=position, theta=theta, phi=phi, t0=jnp.array(0.0))
    log_w, flat_times, flat_indices, total_charge = prediction_simulator(track, key)

    # Charge loss (Poisson NLL)
    charge_loss = poisson_nll(observed_counts, total_charge)

    # Time loss (First-arrival NLL with shifted observations)
    t_obs_shifted = observed_times - t0
    time_nll = first_arrival_nll(
        log_w, flat_times, flat_indices,
        t_obs_shifted, TAU_TIME, NUM_DETECTORS)

    hit_mask = observed_counts > 0
    n_hit = jnp.sum(hit_mask) + 1e-8
    time_loss = jnp.sum(jnp.where(hit_mask, time_nll, 0.0)) / n_hit

    # Vertex loss with dynamic tau_vtx
    tau_vtx = jax.lax.stop_gradient(
        TAU_VTX_PARAM_A * NRAYS_FLOAT + TAU_VTX_PARAM_B * energy + TAU_VTX_PARAM_C
    )
    tau_vtx = jnp.clip(tau_vtx, 0.05, 0.95)

    vertex_loss = origin_time_loss_configurable(
        jax.lax.stop_gradient(position), detector_points,
        observed_times, observed_counts, t0, tau=tau_vtx
    )

    # 3-term combined loss
    c, t, v, s = charge_loss, time_loss, vertex_loss, 0.
    combined = (jnp.sqrt((c + s) * (t + s) * (v + s)) +
                jnp.sqrt((c + s) * jax.lax.stop_gradient((t + s) * (v + s))) +
                jnp.sqrt((v + s) * jax.lax.stop_gradient((t + s) * (c + s))))

    return combined, (charge_loss, time_loss, vertex_loss)


combined_grad_fn = jit(value_and_grad(combined_product_loss, has_aux=True))

config_dir = Path('../s3df_jobs/nrays_config')
config_path = config_dir / 'opt_config_4.json'
script_path = config_dir / 'create_configs.py'

if not config_path.exists():
    print("Config file not found. Creating it...")
    subprocess.run(
        [sys.executable, script_path.name],
        cwd=config_dir,
        check=True
    )
    print("Config file successfully created.")
else:
    print("Config file already exists.")

adam_config = load_config(config_path)

detector_bounds = get_detector_bounds(detector)

print("Combined product loss function defined (likelihood-based)")

In [ ]:
data_file = '../data/water/muon/muon_gun_1050_MeV_100_events_fixed_energy.root'
# Select entry from ROOT file
entry_idx = 0

# Load photon data from ROOT file
photon_data = read_photon_data_from_photonsim(data_file, entry_idx)

# Process photon data
photon_origins = photon_data['photon_origins']
photon_directions = photon_data['photon_directions']
photon_times = photon_data['photon_times']
N = len(photon_origins)

# the number 1_000_000 is hard coded also in _simulation_core
padding_size = max(0, 1_000_000-N)

# Pad the origins array (2D array with shape [N,3])
photon_data['photon_origins'] = jnp.pad(photon_origins, ((0, padding_size), (0, 0)), 
                                    mode='constant', constant_values=0)

# Pad the directions array with a default unit vector [0,0,1]
default_direction = jnp.array([0.0, 0.0, 1.0])
padding_directions = jnp.tile(default_direction, (padding_size, 1))
if padding_size > 0:
    photon_data['photon_directions'] = jnp.concatenate([photon_directions, padding_directions], axis=0)
else:
    photon_data['photon_directions'] = photon_directions

# Pad the times array (1D array with shape [N])
photon_data['photon_times'] = jnp.pad(photon_times, (0, padding_size),
                                      mode='constant', constant_values=0)

photon_data['N'] = N

# Generate random track parameters
key = jax.random.PRNGKey(123)


# Random position within detector bounds (60% of full volume)
fraction = 0.6

if hasattr(detector, "H"):  # Cylinder
    # Cylindrical coordinates
    r_vert = jax.random.uniform(key, shape=(), minval=0, maxval=detector.r * fraction)
    key, _ = jax.random.split(key)
    theta_pos = jax.random.uniform(key, shape=(), minval=0, maxval=2*jnp.pi)
    key, _ = jax.random.split(key)
    z_vert = jax.random.uniform(key, shape=(), 
                                minval=-detector.H/2 * fraction, 
                                maxval=detector.H/2 * fraction)
    true_position = jnp.array([
        r_vert * jnp.cos(theta_pos),
        r_vert * jnp.sin(theta_pos),
        z_vert
    ])

elif hasattr(detector, "r"):  # Sphere
    # Uniform random point inside a sphere (not just uniform radius!)
    key, subkey1 = jax.random.split(key)
    u = jax.random.uniform(subkey1, shape=())
    r_vert = detector.r * (u ** (1/3)) * fraction  # cube root for uniform volume

    key, subkey2 = jax.random.split(key)
    theta_pos = jax.random.uniform(subkey2, shape=(), minval=0, maxval=2*jnp.pi)
    key, subkey3 = jax.random.split(key)
    cos_phi = jax.random.uniform(subkey3, shape=(), minval=-1, maxval=1)
    sin_phi = jnp.sqrt(1 - cos_phi**2)

    true_position = jnp.array([
        r_vert * sin_phi * jnp.cos(theta_pos),
        r_vert * sin_phi * jnp.sin(theta_pos),
        r_vert * cos_phi
    ])
else:
    raise ValueError("Unknown detector geometry: must be Sphere or Cylinder")

# Random direction
key, _ = jax.random.split(key)
phi = jax.random.uniform(key, shape=(), minval=0, maxval=2*jnp.pi)
key, _ = jax.random.split(key)
cos_theta = jax.random.uniform(key, shape=(), minval=-1, maxval=1)
sin_theta = jnp.sqrt(1 - cos_theta**2)
true_direction = jnp.array([sin_theta * jnp.cos(phi), sin_theta * jnp.sin(phi), cos_theta])

# Use energy from ROOT file
true_energy = photon_data['energy']  # Fixed energy

# Create particle parameters
true_track = ParticleParams.from_cartesian(energy=true_energy, position=true_position, direction=true_direction, t0=0.0)

# Compute rotation to transform from original direction (0,0,1) to true_direction
original_direction = jnp.array([0.0, 0.0, 1.0])
true_direction_norm = true_direction / (jnp.linalg.norm(true_direction) + 1e-8)

# Rotation axis = cross product of original and target directions
rotation_axis = jnp.cross(original_direction, true_direction_norm)
axis_norm = jnp.linalg.norm(rotation_axis)

# Handle case where directions are parallel (axis_norm ~ 0)
rotation_axis = jnp.where(
    axis_norm < 1e-6,
    jnp.array([1.0, 0.0, 0.0]),  # Arbitrary axis when parallel
    rotation_axis / (axis_norm + 1e-8)
)

# Rotation angle = arccos of dot product
rotation_angle = jnp.arccos(jnp.clip(
    jnp.dot(original_direction, true_direction_norm), -1.0, 1.0
))

# Set rotation parameters in photon_data
photon_data['rotation_axis'] = rotation_axis
photon_data['rotation_angle'] = rotation_angle
photon_data['apply_rotation'] = jnp.array(True)

# Set translation parameters to move from origin to true_position
photon_data['apply_translation'] = jnp.array(True)
photon_data['translation_vector'] = true_position

# Generate data-like event
key, _ = jax.random.split(key)
true_data = jax.lax.stop_gradient(data_simulator(true_track, key, photon_data))

# Convert true direction (cartesian → spherical)
x, y, z = true_direction
true_theta = np.arccos(z)
true_phi = np.arctan2(y, x)

key, _ = jax.random.split(key)
true_t0 = jax.random.uniform(key, shape=(), minval=-15.0, maxval=15.0)

hit_counts, hit_times = true_data
hit_times += true_t0
true_data = [hit_counts, hit_times]

In [ ]:
def run_complete_optimization_adam_from_guess(
    initial_params,  # [x, y, z, t0, theta, phi, energy]
    observed_times,
    observed_counts,
    true_energy,
    true_position,
    true_direction,
    TRUE_T0,
    config,
    detector_bounds,
    prediction_simulator,
    combined_grad_fn,
    verbosity=2
):
    """
    Run Adam optimization starting from a provided initial guess.
    Uses likelihood-based loss (Poisson NLL + first-arrival NLL + vertex loss).
    """

    import jax
    import jax.numpy as jnp
    import numpy as np
    import time
    from jax import value_and_grad
    import optax
    from lucid.optimization.utils.functions import spherical_to_cartesian, cartesian_to_spherical

    # Unpack true spherical coords for error metrics
    true_theta, true_phi = cartesian_to_spherical(true_direction)

    # Extract optimizer configuration
    ADAM_LEARNING_RATE = config['adam_optimizer']['learning_rate']*5.
    ADAM_B1 = config['adam_optimizer']['b1']
    ADAM_B2 = config['adam_optimizer']['b2']
    ADAM_EPS = config['adam_optimizer']['eps']
    damping_factor = config['optimization_params'].get('damping_factor', 0.998)
    tolerance = 1e-6

    # Learning-rate scaling
    POS_LR_SCALE = config['learning_rates']['position_learning_rate']*5.
    DIR_LR_SCALE = config['learning_rates']['direction_learning_rate']*0.2
    T0_LR_SCALE = config['learning_rates']['t0_learning_rate']*0.2
    ENE_LR_SCALE = config['learning_rates']['energy_learning_rate']*0.2
    update_scales = jnp.array([
        POS_LR_SCALE, POS_LR_SCALE, POS_LR_SCALE,  # X,Y,Z
        T0_LR_SCALE, DIR_LR_SCALE, DIR_LR_SCALE,   # t0, theta, phi
        ENE_LR_SCALE                               # energy
    ])

    # Detector constraints
    DETECTOR_R = detector_bounds.get('r', None)
    DETECTOR_H = detector_bounds.get('H', None)

    if verbosity >= 2:
        print("\nStarting Adam optimization from provided initial parameters:")
        print(f"  Initial params: {initial_params}")
        print(f"  True position: {true_position}")
        print(f"  True direction: {true_direction}")
        print(f"  True energy: {true_energy:.1f} MeV, True t0: {TRUE_T0:.3f}")

    # Initialize Adam optimizer
    optimizer = optax.adam(learning_rate=ADAM_LEARNING_RATE, b1=ADAM_B1, b2=ADAM_B2, eps=ADAM_EPS)
    opt_state = optimizer.init(initial_params)
    current_params = jnp.array(initial_params)

    history = {
        'parameters': [current_params.copy()],
        'combined_losses': [],
        'charge_losses': [],
        'time_losses': [],
        'vertex_losses': [],
        'position_errors': [],
        'direction_errors': [],
        't0_errors': [],
        'energy_errors': [],
    }

    opt_key = jax.random.PRNGKey(12345)
    current_damping_w = 5.0

    adam_start_time = time.time()

    for iteration in range(1000):
        opt_key, _ = jax.random.split(opt_key)

        (combined_loss, (charge_loss_val, time_loss_val, vertex_loss_val)), grad = combined_grad_fn(
            current_params, observed_times, observed_counts, opt_key
        )

        if jnp.any(jnp.isnan(grad)):
            grad = jnp.nan_to_num(grad, nan=0.0)

        grad_norm = jnp.linalg.norm(grad)
        if grad_norm < tolerance:
            break

        updates, opt_state = optimizer.update(grad, opt_state, current_params)
        current_damping_w *= damping_factor
        scaled_updates = updates * update_scales * current_damping_w

        # Apply updates
        current_params = optax.apply_updates(current_params, scaled_updates)

        # Clip within detector bounds
        if DETECTOR_R is not None and DETECTOR_H is not None:
            current_params = jnp.array([
                jnp.clip(current_params[0], -DETECTOR_R * 0.95, DETECTOR_R * 0.95),
                jnp.clip(current_params[1], -DETECTOR_R * 0.95, DETECTOR_R * 0.95),
                jnp.clip(current_params[2], -DETECTOR_H/2 * 0.95, DETECTOR_H/2 * 0.95),
                jnp.clip(current_params[3], -20.0, 20.0),
                current_params[4],
                current_params[5],
                jnp.clip(current_params[6], 300.0, 2000.0)
            ])

        # Calculate derived quantities
        current_position = current_params[:3]
        current_t0 = current_params[3]
        current_theta = current_params[4]
        current_phi = current_params[5]
        current_energy = current_params[6]
        current_direction = spherical_to_cartesian(current_theta, current_phi)

        # Errors
        position_error = float(jnp.linalg.norm(current_position - true_position))
        energy_error = float(abs(current_energy - true_energy))
        t0_error = float(abs(current_t0 - TRUE_T0))
        cos_angle = np.clip(np.dot(np.array(current_direction), np.array(true_direction)), -1.0, 1.0)
        direction_error = float(np.degrees(np.arccos(cos_angle)))

        # Store
        history['parameters'].append(current_params.copy())
        history['combined_losses'].append(float(combined_loss))
        history['charge_losses'].append(float(charge_loss_val))
        history['time_losses'].append(float(time_loss_val))
        history['vertex_losses'].append(float(vertex_loss_val))
        history['position_errors'].append(position_error)
        history['direction_errors'].append(direction_error)
        history['t0_errors'].append(t0_error)
        history['energy_errors'].append(energy_error)

        if verbosity >= 2 and ((iteration + 1) % 100 == 0 or iteration == 0):
            print(f"  Iter {iteration}: loss={combined_loss:.6f}, grad_norm={grad_norm:.4f}, "
                  f"pos_err={position_error:.3f} m, dir_err={direction_error:.2f}°, "
                  f"t0_err={t0_error:.3f}, E_err={energy_error:.1f}")

    adam_end_time = time.time()

    # Final state
    final_position = current_params[:3]
    final_t0 = current_params[3]
    final_theta = current_params[4]
    final_phi = current_params[5]
    final_energy = current_params[6]
    final_direction = spherical_to_cartesian(final_theta, final_phi)

    cos_angle_final = np.clip(np.dot(np.array(final_direction), np.array(true_direction)), -1.0, 1.0)
    final_direction_error = float(np.degrees(np.arccos(cos_angle_final)))
    final_position_error = float(jnp.linalg.norm(final_position - true_position))
    final_energy_error = float(abs(final_energy - true_energy))
    final_t0_error = float(abs(final_t0 - TRUE_T0))

    return {
        'initial_params': np.array(initial_params),
        'final_params': np.array(current_params),
        'final_position': np.array(final_position),
        'final_direction': np.array(final_direction),
        'final_theta': float(final_theta),
        'final_phi': float(final_phi),
        'final_t0': float(final_t0),
        'final_energy': float(final_energy),
        'final_position_error': final_position_error,
        'final_direction_error': final_direction_error,
        'final_t0_error': final_t0_error,
        'final_energy_error': final_energy_error,
        'adam_optimization_time': adam_end_time - adam_start_time,
        'history': history,
        'converged': grad_norm < tolerance,
    }


import numpy as np

def generate_hybrid_initial_guess(
    true_params,
    detector_bounds,
    angle_sigma_deg=5.0,     # angular spread around true direction
    t0_bounds=(-3.0, 3.0),   # ns
    energy_bounds=(500.0, 1500.0),  # MeV
    seed=None
):
    """
    Generate a hybrid random initial guess:
    - Random position (within 0.8× detector volume)
    - Random t0 and energy within given ranges
    - Angles (theta, phi) are Gaussian around true values

    Args:
        true_params (array-like): [X, Y, Z, t0, theta, phi, energy]
        detector_bounds (dict): {'r': ..., 'H': ...} cylindrical bounds
        angle_sigma_deg (float): angular std. dev. around true angles (degrees)
        t0_bounds (tuple): (min, max) for uniform t0 (ns)
        energy_bounds (tuple): (min, max) for uniform energy (MeV)
        seed (int, optional): random seed for reproducibility

    Returns:
        np.ndarray: [X, Y, Z, t0, theta, phi, energy]
    """
    rng = np.random.default_rng(seed)

    # Unpack true values
    _, _, _, true_t0, true_theta, true_phi, _ = np.array(true_params, dtype=float)

    # --- Detector limits ---
    R = detector_bounds.get('r', 10.0)
    H = detector_bounds.get('H', 20.0)
    R_lim = 0.5 * R
    H_lim = 0.5 * (H / 2.0)

    # Random position (uniform in cylinder volume)
    r = R_lim * np.sqrt(rng.uniform(0, 1))
    phi_pos = rng.uniform(0, 2 * np.pi)
    x = r * np.cos(phi_pos)
    y = r * np.sin(phi_pos)
    z = rng.uniform(-H_lim, H_lim)

    # --- Random t0 ---
    t0 = rng.uniform(*t0_bounds)

    # --- Direction near true ---
    theta_sigma = np.radians(angle_sigma_deg)
    phi_sigma = np.radians(angle_sigma_deg)
    theta = np.clip(true_theta + rng.normal(0, theta_sigma), 0, np.pi)
    phi = np.mod(true_phi + rng.normal(0, phi_sigma), 2 * np.pi)

    # --- Random energy ---
    energy = rng.uniform(*energy_bounds)

    return np.array([x, y, z, t0, theta, phi, energy])

In [ ]:
true_params = np.array([
    true_position[0],
    true_position[1],
    true_position[2],
    true_t0,
    true_theta,
    true_phi,
    true_energy
])

all_event_results = []
for i in range(5):
    initial_guess = generate_hybrid_initial_guess(true_params, detector_bounds,  angle_sigma_deg=15, seed=45+i)

    print("True params:    ", np.round(true_params, 3))
    print("Initial guess:  ", np.round(initial_guess, 3))

    results_from_guess = run_complete_optimization_adam_from_guess(
        initial_params=initial_guess,
        observed_times=true_data[1],
        observed_counts=true_data[0],
        true_energy=true_energy,
        true_position=true_position,
        true_direction=true_direction,
        TRUE_T0=true_t0,
        config=adam_config,
        detector_bounds=detector_bounds,
        prediction_simulator=prediction_simulator,
        combined_grad_fn=combined_grad_fn,
    )
    all_event_results.append(results_from_guess)

In [ ]:
# =====================================================================
# Reconstruct parameter evolution (from optimization history)
# =====================================================================
def extract_histories(all_event_results):
    """
    Collect all optimization histories from multiple initial guesses.
    Compatible with results returned by run_complete_optimization_adam_from_guess().
    """
    h = {'position': [], 'direction': [], 'energy': [], 't0': []}

    for ev in all_event_results:
        hist = ev['history']
        params = np.array(hist['parameters'])  # shape (n_iter, 7)
        # Columns: [x, y, z, t0, theta, phi, energy]
        h['position'].append(params[:, :3])
        h['t0'].append(params[:, 3])
        h['direction'].append(params[:, 4:6])  # theta, phi
        h['energy'].append(params[:, 6])

    # Convert lists to numpy arrays
    for k in h:
        h[k] = np.array(h[k])  # (n_events, n_iterations, ...)
    return h


histories = extract_histories(all_event_results)


def reconstruct_reco_parameters(all_event_results, histories, true_position, true_direction, true_energy, true_t0):
    """
    Build reco parameter arrays aligned with the true parameters.
    Includes true values as well as reconstructed and difference arrays.
    Works for multiple events (one per initial guess) or just one.
    """
    n_events, n_iterations = histories['t0'].shape[:2]

    # Convert true direction to spherical
    x, y, z = true_direction
    true_theta = np.arccos(z)
    true_phi = np.arctan2(y, x)

    # Tile true values for vectorized reconstruction
    true_pos = np.tile(true_position, (n_iterations, 1))
    true_pos = np.expand_dims(true_pos, 0).repeat(n_events, axis=0)
    true_theta_arr = np.full((n_events, n_iterations), true_theta)
    true_phi_arr = np.full((n_events, n_iterations), true_phi)
    true_energy_arr = np.full((n_events, n_iterations), true_energy)
    true_t0_arr = np.full((n_events, n_iterations), true_t0)

    # Construct differences (reco - true)
    position_diff = histories['position'] - true_pos
    direction_diff = histories['direction'] - np.stack([true_theta_arr, true_phi_arr], axis=-1)
    energy_diff = histories['energy'] - true_energy_arr
    t0_diff = histories['t0'] - true_t0_arr

    reco = {
        # Reconstructed values
        'position': histories['position'],
        'theta': histories['direction'][:, :, 0],
        'phi': histories['direction'][:, :, 1],
        'energy': histories['energy'],
        't0': histories['t0'],

        # True values (added)
        'true_position': true_pos,
        'true_theta': true_theta_arr,
        'true_phi': true_phi_arr,
        'true_energy': true_energy_arr,
        'true_t0': true_t0_arr,

        # Differences (reco - true)
        'position_diff': position_diff,
        'direction_diff': direction_diff,
        'energy_diff': energy_diff,
        't0_diff': t0_diff
    }
    return reco



# Construct reco trajectories relative to true parameters
reco_data = reconstruct_reco_parameters(
    all_event_results,
    histories,
    true_position=true_position,
    true_direction=true_direction,
    true_energy=true_energy,
    true_t0=true_t0
)

print("Reconstructed reco parameter trajectories.")





In [ ]:
import numpy as np
import plotly.graph_objects as go
import os
from lucid.utils import load_range_params

from lucid.optimization.utils.geometry import (
    create_cylinder_surface,
    create_sphere_surface,
    create_box_surface,
)


def spherical_to_cartesian(theta, phi):
    """Convert spherical angles to Cartesian direction vector."""
    return np.array([
        np.sin(theta) * np.cos(phi),
        np.sin(theta) * np.sin(phi),
        np.cos(theta)
    ], dtype=float)


def visualize_tracks_at_iterations_3D(
    reco_data,
    range_params,
    detector_bounds,
    detector_name="Detector",
    figures_dir=None,
    move_threshold=0.50,  # meters → 25 cm
    max_iter_gap=3000      # fallback: every 10 iterations
):
    """
    Show detector surface and reconstructed tracks at selected iterations.
    A new arrow is drawn whenever the reconstructed vertex position moves
    by >= move_threshold (in meters) from the previous arrow position,
    or after max_iter_gap iterations (whichever happens first).
    Adds a large line for the final reconstructed iteration of each event.
    """

    # --- Helper: energy→range conversion ---
    def calculate_particle_range(energy_mev, range_params):
        a = range_params['parameters']['a']
        b = range_params['parameters']['b']
        return (a * energy_mev + b) / 1000.0  # mm → m

    # --- True values ---
    true_pos = np.mean(reco_data['true_position'][0], axis=0)
    true_theta = np.mean(reco_data['true_theta'][0])
    true_phi = np.mean(reco_data['true_phi'][0])
    true_E = np.mean(reco_data['true_energy'][0])
    true_dir = spherical_to_cartesian(true_theta, true_phi)
    true_range = calculate_particle_range(true_E, range_params)

    # --- Choose iteration indices automatically based on movement ---
    pos_traj = np.asarray(reco_data["position"][0])  # use first event
    iteration_indices = [0]
    last_pos = pos_traj[0]
    last_added = 0

    for i in range(1, len(pos_traj)):
        move_dist = np.linalg.norm(pos_traj[i] - last_pos)
        if move_dist >= move_threshold or (i - last_added) >= max_iter_gap:
            iteration_indices.append(i)
            last_pos = pos_traj[i]
            last_added = i

    print(f"Selected {len(iteration_indices)} iterations: {iteration_indices[:10]}{'...' if len(iteration_indices) > 10 else ''}")

    # --- Figure setup ---
    fig = go.Figure()

    # --- Detector surface ---
    det_type = detector_bounds.get("type", "cylinder")
    if det_type == "cylinder":
        r = detector_bounds["r"]
        H = detector_bounds["H"]
        x, y, z = create_cylinder_surface(r, H)
        fig.add_trace(go.Surface(
            x=x, y=y, z=z,
            surfacecolor=np.ones_like(x),
            colorscale=[[0, "lightgrey"], [1, "lightgrey"]],
            opacity=0.15, showscale=False,
            name="Detector Surface", hoverinfo="skip"
        ))
    elif det_type == "sphere":
        r = detector_bounds["r"]
        x, y, z = create_sphere_surface(r)
        fig.add_trace(go.Surface(
            x=x, y=y, z=z,
            surfacecolor=np.ones_like(x),
            colorscale=[[0, "lightgrey"], [1, "lightgrey"]],
            opacity=0.15, showscale=False,
            name="Detector Surface", hoverinfo="skip"
        ))
    elif det_type == "box":
        xlen, ylen, zlen = detector_bounds["x"], detector_bounds["y"], detector_bounds["z"]
        vertices, edges = create_box_surface(xlen, ylen, zlen)
        vertices = np.asarray(vertices)
        for edge in edges:
            v = vertices[list(edge)]
            fig.add_trace(go.Scatter3d(
                x=v[:, 0], y=v[:, 1], z=v[:, 2],
                mode="lines", line=dict(color="gray", width=2),
                showlegend=False, hoverinfo="skip"
            ))

    # --- Reconstructed tracks ---
    n_events = len(reco_data["t0"])
    colors = ["red", "cyan", "lime", "orange", "magenta", "deepskyblue"]

    for event_id in range(n_events):
        pos_traj = np.asarray(reco_data["position"][event_id])
        theta_traj = np.asarray(reco_data["theta"][event_id])
        phi_traj = np.asarray(reco_data["phi"][event_id])
        energy_traj = np.asarray(reco_data["energy"][event_id])

        color = colors[event_id % len(colors)]

        # Intermediate iteration arrows
        for step in iteration_indices:
            if step >= len(pos_traj):
                continue

            pos = pos_traj[step]
            theta = theta_traj[step]
            phi = phi_traj[step]
            energy = energy_traj[step]

            direction = spherical_to_cartesian(theta, phi)
            direction /= np.linalg.norm(direction)
            length = calculate_particle_range(energy, range_params)
            end = pos + direction * length

            fig.add_trace(go.Scatter3d(
                x=[pos[0], end[0]],
                y=[pos[1], end[1]],
                z=[pos[2], end[2]],
                mode="lines",
                line=dict(color=color, width=4),
                showlegend=False
            ))

            # Arrow origin marker
            fig.add_trace(go.Scatter3d(
                x=[pos[0]], y=[pos[1]], z=[pos[2]],
                mode="markers",
                marker=dict(size=4, color=color, symbol="circle"),
                showlegend=False
            ))

        # --- Final iteration (big arrow) ---
        final_idx = len(pos_traj) - 1
        pos = pos_traj[final_idx]
        theta = theta_traj[final_idx]
        phi = phi_traj[final_idx]
        energy = energy_traj[final_idx]

        direction = spherical_to_cartesian(theta, phi)
        direction /= np.linalg.norm(direction)
        length = calculate_particle_range(energy, range_params)
        end = pos + direction * length * 2

        # Each event gets a 'Last iteration' arrow (with legend)
        fig.add_trace(go.Scatter3d(
            x=[pos[0], end[0]],
            y=[pos[1], end[1]],
            z=[pos[2], end[2]],
            mode="lines",
            line=dict(color=color, width=12),
            name=f"Last iteration (initial guess {event_id})",
            legendgroup="last",            # 👈 group legend entries
            showlegend=True
        ))

    # --- True track (blue line) ---
    true_end = true_pos + true_dir * true_range * 2
    fig.add_trace(go.Scatter3d(
        x=[true_pos[0], true_end[0]],
        y=[true_pos[1], true_end[1]],
        z=[true_pos[2], true_end[2]],
        mode="lines",
        line=dict(color="blue", width=20),
        name="True track",
        legendgroup="true",         # 👈 group for true track
        showlegend=True
    ))


    # --- Clean white background (no axes or grid) ---
    fig.update_layout(
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            bgcolor="white"
        ),
        paper_bgcolor="white",
        plot_bgcolor="white",
        width=1100,
        height=850,
        margin=dict(l=0, r=0, t=0, b=0),
        showlegend=True,
        legend=dict(
            x=0.02, y=0.98,
            bgcolor="rgba(255,255,255,0.8)",
            bordercolor="gray",
            borderwidth=1
        )
    )

    # --- Save ---
    filename = f"{detector_name}_tracks_iterations_3D.html"
    if figures_dir:
        os.makedirs(figures_dir, exist_ok=True)
        filename = os.path.join(figures_dir, filename)
    fig.write_html(filename)
    print(f"3D tracks visualization saved to {filename}")

    return fig, filename


In [ ]:
range_params = load_range_params('muon', 'water')

fig, filename = visualize_tracks_at_iterations_3D(
    reco_data=reco_data,
    range_params=range_params,
    detector_bounds=get_detector_bounds(detector),
    detector_name=detector_name,
    figures_dir="figures"
)

In [ ]:
from IPython.display import HTML

HTML(filename=f"figures/{detector_name}_tracks_iterations_3D.html")

# Alternatively, if one has an output file from reconstruction

In [ ]:
import os
import sys
sys.path.append('..')
import pickle
import torch

figures_dir = 'figures/'
os.makedirs(figures_dir, exist_ok=True)
results_file = '/sdf/data/neutrino/cjesus/lucid_output/paper_files/results_50k_nrays.pkl'

with open(results_file, 'rb') as f:
    results_summary = pickle.load(f)

all_event_results = results_summary['all_event_results']

In [ ]:
from lucid.optimization.utils.geometry import compute_cone_cylinder_intersection, create_cylinder_surface
from lucid.geometry import generate_detector
import jax.numpy as jnp
from lucid.optimization.grid_search import get_detector_bounds
from lucid.optimization.utils.visualization import create_event_3D_visualization

json_filename = '../config/SK_geom_config.json'
detector = generate_detector(json_filename)
detector_points = jnp.array(detector.all_points)
detector_bounds = get_detector_bounds(detector)

event_ID = 0
true_charges = all_event_results[event_ID]['event_data']['true_data'][0]
true_times = all_event_results[event_ID]['event_data']['true_data'][1]

fig = create_event_3D_visualization(event_ID, all_event_results, detector_points, \
                                                         true_charges, true_times, detector_bounds, color_by='charge', min_charge=2.5, figures_dir='figures/', detector_name='SK')
fig.show()

In [ ]:
from lucid.optimization.utils.visualization import create_optimization_path_3d_visualization
event_ID = 0
print(f"\nCreating optimization_path_3d_visualization for Event {event_ID}...")
fig = create_optimization_path_3d_visualization(event_ID, all_event_results, arrow_every_n=100, figures_dir='figures/', detector_name='SK')
fig.show()